# BrokerFlow - Notebook de test broker 



In [ ]:
import json
from copy import deepcopy
from datetime import datetime

import pandas as pd
from fastapi.testclient import TestClient

from src.api.main import create_app
from src.config.settings import settings

pd.set_option("display.max_colwidth", 120)

app = create_app()
client = TestClient(app)

print("API version:", settings.api_version)
print("AGENT_LLM_ENABLED (current):", settings.agent_llm_enabled)

: 

In [ ]:
BASE_APP = {
    "application_id": "APP-DEMO",
    "customer_id": "CUST-DEMO",
    "snapshot_date": "2025-01-10",
    "employment_status": "employed",
    "monthly_income": 3500,
    "requested_amount": 12000,
    "debt_to_income_ratio": 0.30,
    "prior_late_payments": 0,
    "has_prior_default": 0,
    "free_text_note": "Stable profile and regular income.",
    "documents": []
}

def make_case(case_id: str, updates: dict) -> dict:
    app = deepcopy(BASE_APP)
    app.update(updates)
    app["application_id"] = case_id
    app["customer_id"] = case_id.replace("APP", "CUST")
    return app

def score_case(payload: dict) -> dict:
    resp = client.post("/v2/score", json=payload)
    resp.raise_for_status()
    return resp.json()

def review_case(payload: dict) -> list:
    resp = client.post("/v1/review-detailed", json=payload)
    resp.raise_for_status()
    return resp.json()

: 

In [ ]:
broker_cases = [
    make_case(
        "APP-BROKER-001",
        {
            "free_text_note": "Client wants to finance studies.",
            "requested_amount": 8000,
            "debt_to_income_ratio": 0.20
        },
    ),
    make_case(
        "APP-BROKER-002",
        {
            "free_text_note": "Besoin urgent, dossier incomplet, pieces manquantes.",
            "monthly_income": 1600,
            "requested_amount": 22000,
            "debt_to_income_ratio": 0.70,
            "prior_late_payments": 2,
            "documents": [
                {
                    "document_id": "DOC-2-1",
                    "application_id": "APP-BROKER-002",
                    "document_type": "income_proof",
                    "is_required": True,
                    "is_provided": False
                }
            ]
        },
    ),
    make_case(
        "APP-BROKER-003",
        {
            "employment_status": "unemployed",
            "monthly_income": 1200,
            "requested_amount": 15000,
            "free_text_note": "Broker note says stable job and no issue."
        },
    ),
    make_case(
        "APP-BROKER-004",
        {
            "prior_late_payments": 3,
            "free_text_note": "Aucun retard de paiement selon le client."
        },
    ),
    make_case(
        "APP-BROKER-005",
        {
            "free_text_note": "Perhaps there was an incident, a verifier."
        },
    ),
]

print(f"Nombre de dossiers broker: {len(broker_cases)}")

: 

In [ ]:
# Option: forcer mode deterministic pour un benchmark stable
# settings.agent_llm_enabled = False

results = []
for payload in broker_cases:
    pred = score_case(payload)
    results.append(
        {
            "application_id": pred["application_id"],
            "risk_score": pred["risk_score"],
            "risk_class": pred["risk_class"],
            "recommendation": pred["recommendation"],
            "alerts_count": len(pred.get("alerts", [])),
            "alert_severity": pred.get("decision_alert_severity"),
            "completeness": pred.get("completeness"),
            "summary": pred.get("summary", "")
        }
    )

df = pd.DataFrame(results).sort_values(by=["risk_score"], ascending=False)
df

: 

In [ ]:
# Affichage broker-friendly
view = df.copy()
view["risk_score"] = view["risk_score"].map(lambda x: f"{x:.4f}")
view["completeness"] = view["completeness"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")
view

: 

In [ ]:
# Zoom detaille sur un dossier
focus_id = "APP-BROKER-002"
focus_payload = [x for x in broker_cases if x["application_id"] == focus_id][0]
focus_pred = score_case(focus_payload)
focus_review = review_case(focus_payload)

print("=== Decision ===")
print("application_id:", focus_pred["application_id"])
print("risk_score:", round(focus_pred["risk_score"], 4))
print("risk_class:", focus_pred["risk_class"])
print("recommendation:", focus_pred["recommendation"])
print("decision_reason_codes:", focus_pred.get("decision_reason_codes"))
print("decision_alert_severity:", focus_pred.get("decision_alert_severity"))
print("decision_threshold:", focus_pred.get("decision_threshold"))
print()
print("=== Top factors ===")
for f, v in focus_pred.get("top_factors", []):
    print(f"- {f}: {v:.4f}")
print()
print("=== Alerts structured (/v1/review-detailed) ===")
for item in focus_review:
    print(f"- [{item.get('severity','?')}] {item.get('code','?')} | {item.get('message','')}")
print()
print("=== Summary ===")
print(focus_pred.get("summary", ""))

: 

## Test 

Si tu veux tester comme en prod:

1. Lance l'API dans un terminal:

```bash
uvicorn src.api.main:app --reload --port 8000
```

2. Execute la cellule suivante pour appeler `http://127.0.0.1:8000/v2/score`.


import requests

live_payload = broker_cases[1]
url = "http://127.0.0.1:8000/v2/score"

try:
    r = requests.post(url, json=live_payload, timeout=20)
    print("status:", r.status_code)
    if r.ok:
        live_pred = r.json()
        print(json.dumps({
            "application_id": live_pred.get("application_id"),
            "risk_score": live_pred.get("risk_score"),
            "risk_class": live_pred.get("risk_class"),
            "recommendation": live_pred.get("recommendation")
        }, indent=2))
    else:
        print(r.text)
except Exception as e:
    print("Serveur non disponible ou erreur reseau:", e)
# Export des resultats pour partage interne broker/compliance
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"data/processed/broker_demo_results_{timestamp}.csv"
json_path = f"data/processed/broker_demo_results_{timestamp}.json"

df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("CSV:", csv_path)
print("JSON:", json_path)